# Cross-Attention version —— generation & fusion

cross-attn fusion is more "compositional": **concatenate the text tokens** of the two, so the image does cross-attention over both sets of words at once.
Requires CLIP (to encode free text on the fly).

## Model + diffusion (consistent with training)

In [ ]:
import math, os, glob, json, random
import torch
import torch.nn as nn
import torch.nn.functional as F
import os as _os
_os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")
device = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")

def sinusoidal_embedding(t, dim):
    half = dim // 2
    freqs = torch.exp(-math.log(10000) * torch.arange(half, device=t.device).float() / half)
    args = t[:, None].float() * freqs[None, :]
    return torch.cat([torch.sin(args), torch.cos(args)], dim=-1)

class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, c_dim, groups=8):
        super().__init__()
        self.block1 = nn.Sequential(nn.GroupNorm(groups, in_ch), nn.SiLU(), nn.Conv2d(in_ch, out_ch, 3, padding=1))
        self.cond_mlp = nn.Sequential(nn.SiLU(), nn.Linear(c_dim, out_ch))
        self.block2 = nn.Sequential(nn.GroupNorm(groups, out_ch), nn.SiLU(), nn.Conv2d(out_ch, out_ch, 3, padding=1))
        self.skip = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()
    def forward(self, x, c):
        h = self.block1(x); h = h + self.cond_mlp(c)[:, :, None, None]; h = self.block2(h)
        return h + self.skip(x)

class SelfAttn(nn.Module):
    """pixel <-> pixel self-attention (handles global coherence)."""
    def __init__(self, ch, heads=4):
        super().__init__(); self.norm = nn.GroupNorm(8, ch); self.mha = nn.MultiheadAttention(ch, heads, batch_first=True)
    def forward(self, x):
        B, C, H, W = x.shape
        h = self.norm(x).reshape(B, C, H*W).transpose(1, 2); h, _ = self.mha(h, h, h)
        return x + h.transpose(1, 2).reshape(B, C, H, W)

class CrossAttn(nn.Module):
    """pixel <-> text token cross-attention (spatialized text control): each pixel "queries" every word of the text."""
    def __init__(self, ch, ctx_dim=512, heads=4):
        super().__init__(); self.norm = nn.GroupNorm(8, ch); self.kv = nn.Linear(ctx_dim, ch)
        self.mha = nn.MultiheadAttention(ch, heads, batch_first=True)
    def forward(self, x, ctx, key_padding_mask=None):
        B, C, H, W = x.shape
        q = self.norm(x).reshape(B, C, H*W).transpose(1, 2)   # (B, HW, C) image as query
        kv = self.kv(ctx)                                     # (B, L, C) text tokens as key/value
        h, _ = self.mha(q, kv, kv, key_padding_mask=key_padding_mask)  # mask: True=padding ignored
        return x + h.transpose(1, 2).reshape(B, C, H, W)

class Stage(nn.Module):
    """n_blocks ResBlocks + optional (self-attn -> cross-attn)."""
    def __init__(self, in_ch, out_ch, c_dim, n_blocks=2, attn=False):
        super().__init__()
        self.blocks = nn.ModuleList([ResBlock(in_ch if i==0 else out_ch, out_ch, c_dim) for i in range(n_blocks)])
        self.self_attn  = SelfAttn(out_ch)  if attn else None
        self.cross_attn = CrossAttn(out_ch) if attn else None
    def forward(self, x, c, ctx=None, mask=None):
        for b in self.blocks: x = b(x, c)
        if self.self_attn  is not None: x = self.self_attn(x)
        if self.cross_attn is not None: x = self.cross_attn(x, ctx, mask)
        return x

class UNet(nn.Module):
    """Text-conditioned UNet. Two condition paths: time + pooled text -> global add (ResBlock); text token sequence -> cross-attn (spatial)."""
    def __init__(self, in_ch=3, base=128, c_dim=256, n_blocks=2, txt_dim=512):
        super().__init__(); self.c_dim = c_dim
        self.text_proj = nn.Sequential(nn.Linear(txt_dim, c_dim), nn.SiLU(), nn.Linear(c_dim, c_dim))  # pooled -> global condition
        self.time_mlp  = nn.Sequential(nn.Linear(c_dim, c_dim), nn.SiLU(), nn.Linear(c_dim, c_dim))
        self.in_conv = nn.Conv2d(in_ch, base, 3, padding=1)
        self.down1 = Stage(base,    base,   c_dim, n_blocks)
        self.down2 = Stage(base,    base*2, c_dim, n_blocks)
        self.down3 = Stage(base*2,  base*4, c_dim, n_blocks, attn=True)
        self.downsample = nn.AvgPool2d(2)
        self.mid = Stage(base*4, base*4, c_dim, n_blocks, attn=True)
        self.upsample = nn.Upsample(scale_factor=2, mode="nearest")
        self.up3 = Stage(base*4+base*4, base*2, c_dim, n_blocks, attn=True)
        self.up2 = Stage(base*2+base*2, base,   c_dim, n_blocks)
        self.up1 = Stage(base+base,     base,   c_dim, n_blocks)
        self.out = nn.Sequential(nn.GroupNorm(8, base), nn.SiLU(), nn.Conv2d(base, in_ch, 3, padding=1))
    def forward(self, x, t, pooled, tokens, mask):
        c = self.time_mlp(sinusoidal_embedding(t, self.c_dim)) + self.text_proj(pooled)   # time + pooled text (global)
        x = self.in_conv(x)
        s1 = self.down1(x, c); x = self.downsample(s1)
        s2 = self.down2(x, c); x = self.downsample(s2)
        s3 = self.down3(x, c, tokens, mask); x = self.downsample(s3)            # cross-attn @24
        x = self.mid(x, c, tokens, mask)                                       # cross-attn @12
        x = self.upsample(x); x = self.up3(torch.cat([x, s3], 1), c, tokens, mask)  # cross-attn @24
        x = self.upsample(x); x = self.up2(torch.cat([x, s2], 1), c)
        x = self.upsample(x); x = self.up1(torch.cat([x, s1], 1), c)
        return self.out(x)

In [ ]:
class Diffusion:
    def __init__(self, timesteps=1000, beta_start=1e-4, beta_end=0.02, device="cuda"):
        self.T = timesteps; self.device = device
        beta = torch.linspace(beta_start, beta_end, timesteps, device=device)
        self.beta = beta; self.alpha = 1.0 - beta; self.alpha_bar = torch.cumprod(self.alpha, 0)
    def q_sample(self, x0, t, noise):
        ab = self.alpha_bar[t][:, None, None, None]; return ab.sqrt()*x0 + (1-ab).sqrt()*noise
    def loss(self, model, x0, pooled, tokens, mask):
        B = x0.size(0); t = torch.randint(0, self.T, (B,), device=x0.device); noise = torch.randn_like(x0)
        return F.mse_loss(model(self.q_sample(x0, t, noise), t, pooled, tokens, mask), noise)
    @torch.no_grad()
    def sample(self, model, n, pooled, tokens, mask, null, guidance=4.0, img_size=96):
        """pooled(n,512) tokens(n,L,512) mask(n,L); null=(p,t,m) single entry, expanded to n internally."""
        model.eval()
        np_, nt_, nm_ = null
        np_ = np_[None].expand(n, -1); nt_ = nt_[None].expand(n, -1, -1); nm_ = nm_[None].expand(n, -1)
        x = torch.randn(n, 3, img_size, img_size, device=self.device)
        for i in reversed(range(self.T)):
            t = torch.full((n,), i, device=self.device, dtype=torch.long)
            ec = model(x, t, pooled, tokens, mask)             # conditional
            eu = model(x, t, np_, nt_, nm_)                    # unconditional (null)
            pred = eu + guidance * (ec - eu)                   # CFG
            bt, at, abt = self.beta[i], self.alpha[i], self.alpha_bar[i]
            mean = (1/at.sqrt()) * (x - (bt/(1-abt).sqrt()) * pred)
            x = mean + (bt.sqrt()*torch.randn_like(x) if i > 0 else 0.0)
        model.train(); return x

## Load CLIP (per-token) + weights + clip_seq

In [ ]:
import matplotlib.pyplot as plt
from torchvision.utils import make_grid
try:
    import open_clip
except ImportError:
    import subprocess, sys; subprocess.run([sys.executable,"-m","pip","install","-q","open_clip_torch"]); import open_clip
_clip,_,_ = open_clip.create_model_and_transforms("ViT-B-32", pretrained="laion2b_s34b_b79k")
_tok = open_clip.get_tokenizer("ViT-B-32"); _clip = _clip.to(device).eval()

@torch.no_grad()
def clip_seq(text):
    """One sentence of text -> (pooled(1,512), tokens(1,77,512), mask(1,77))"""
    t = _tok([text]).to(device)
    x = _clip.token_embedding(t) + _clip.positional_embedding
    x = _clip.transformer(x, attn_mask=_clip.attn_mask); x = _clip.ln_final(x)
    p = x[torch.arange(1), t.argmax(-1)] @ _clip.text_projection; p = p/p.norm(dim=-1,keepdim=True)
    return p.float(), x.float(), (t == 0)

EXP_NAME = "exp07_stage1_pokemonALL_xattn"        # ← change to your experiment
OUT_DIR  = "."
CKPT     = os.path.join(OUT_DIR, "checkpoints", "ckpt_ep300.pt")   # ← change epoch
names = json.load(open(os.path.join(OUT_DIR,"classes.json"), encoding="utf-8"))
descs = json.load(open("descriptions.json", encoding="utf-8"))
model = UNet(base=128, n_blocks=2).to(device)
model.load_state_dict(torch.load(CKPT, map_location=device)); model.eval()
diff = Diffusion(timesteps=1000, device=device)
null = clip_seq("")
null = (null[0][0], null[1][0], null[2][0])         # single item
print("loaded:", EXP_NAME, "| classes", len(names))

def show(imgs, title=""):
    g = make_grid(((imgs.clamp(-1,1)+1)/2).cpu(), nrow=min(len(imgs),4)).permute(1,2,0).numpy()
    plt.figure(figsize=(8,8)); plt.imshow(g); plt.axis("off"); plt.title(title); plt.show()

## 1. Generate by name / free text
Using this Pokémon's training description (name+type) works best; you can also write freely.

In [ ]:
PROMPT = "Pikachu, a yellow electric-type mouse pokémon, quadruped body, from generation 1"      # or write anything: "a blue fire-type turtle pokémon, upright body"
N = 8; GUIDANCE = 4.0
p, tok, m = clip_seq(PROMPT)
imgs = diff.sample(model, N, p.expand(N,-1), tok.expand(N,-1,-1), m.expand(N,-1), null, guidance=GUIDANCE, img_size=96)
show(imgs, PROMPT[:40])

## 2. Fusion (cross-attn token concatenation)

**Concatenate** the tokens of descriptions A and B, so the image attends to both word sets via cross-attention → a more "compositional" fusion
(More likely to pick up features from both sides than single-vector interpolation). pooled is the average of the two.

In [ ]:
PROMTPT_A = "Squirtle, a blue water-type tiny turtle pokémon, upright body, from generation 1"; 
PROMTPT_B = "Charmander, a red fire-type lizard pokémon, upright body, from generation 1"; 
N = 8; GUIDANCE = 3.0
pa, ta, ma = clip_seq(PROMTPT_A); pb, tb, mb = clip_seq(PROMTPT_B)
tok = torch.cat([ta, tb], dim=1)               # concat tokens (1, 154, 512)
m   = torch.cat([ma, mb], dim=1)
p   = (pa + pb) / 2; p = p / p.norm(dim=-1, keepdim=True)
imgs = diff.sample(model, N, p.expand(N,-1), tok.expand(N,-1,-1), m.expand(N,-1), null, guidance=GUIDANCE, img_size=96)
show(imgs, f"{PROMTPT_A} + {PROMTPT_B} (xattn token)")

## Notes
- cross-attn makes "placing features in the right location" possible, but with small models/small data it is not a qualitative leap, mainly seen after Stage 2 fine-tuning.
- Token-concatenation fusion is a cross-attn-specific technique; if it turns white/blurry, lower GUIDANCE to 2~3.
- Stage 1 covers all Pokémon but is rough; Stage 2's 20 are the clearest.